In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import property_management
import expenses

SOURCE_PATH = "/Volumes/property_catalog/property_management/landing_zone/expenses.csv"
TARGET_SCHEMA = "property_catalog.property_management"

RAW_TABLE = f"{TARGET_SCHEMA}.raw_expenses"
CLEAN_TABLE = f"{TARGET_SCHEMA}.clean_expenses"
REJECTED_TABLE = f"{TARGET_SCHEMA}.rejected_expenses"

In [0]:
expenses_df = property_management.extract_property_management_data(spark, SOURCE_PATH)

# Incrementally load raw expense records
property_management.upsert_property_management_table(spark,expenses_df,RAW_TABLE,key_column="Id")

display(expenses_df)

In [0]:
# Read raw expenses Delta table
expenses_df = spark.table(RAW_TABLE)

transformed_expenses_df = expenses.transform_expenses(expenses_df)

In [0]:
validated_expenses_df = expenses.validate_expenses(transformed_expenses_df)

display(validated_expenses_df)

In [0]:
clean_expenses_df, rejected_expenses_df = expenses.separate_expenses(validated_expenses_df)

print("Clean expenses:")
display(clean_expenses_df)

print("Rejected expenses:")
display(rejected_expenses_df)

In [0]:
expenses_summary = property_management.summarise_property_management_etl(expenses_df, clean_expenses_df, rejected_expenses_df, SOURCE_PATH)

In [0]:
# Remove validation flags from the clean output
clean_expenses_output_df = clean_expenses_df.drop(
    "invalid_expense_id",
    "duplicate_expense_id",
    "invalid_category",
    "invalid_description",
    "invalid_amount",
    "invalid_expense_date",
    "invalid_property_id"
)

# Incrementally load clean expense records
property_management.upsert_property_management_table(spark, clean_expenses_output_df, CLEAN_TABLE, key_column="Id")

# Incrementally load rejected expense records
property_management.upsert_property_management_table(spark, rejected_expenses_df, REJECTED_TABLE, key_column="Id")

print(f"Clean expense records saved to: {CLEAN_TABLE}")
print(f"Rejected expense records saved to: {REJECTED_TABLE}")